#### Create StorageClass

In [ ]:
vim storageclass.yaml

In [ ]:
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: local-storage

provisioner: kubernetes.io/no-provisioner

volumeBindingMode: WaitForFirstConsumer

In [ ]:
kubectl apply -f storageclass.yaml

#### worker-1

In [ ]:
sudo mkdir -p /data/app-storage
sudo chmod 777 /data/app-storage

Create PV For worker-1

In [ ]:
vim pv-worker1.yaml

In [ ]:
apiVersion: v1
kind: PersistentVolume
metadata:
  name: pv-worker1

spec:
  capacity:
    storage: 5Gi

  accessModes:
    - ReadWriteOnce

  persistentVolumeReclaimPolicy: Retain

  storageClassName: local-storage

  local:
    path: /data/app-storage

  nodeAffinity:
    required:
      nodeSelectorTerms:
      - matchExpressions:
        - key: kubernetes.io/hostname
          operator: In
          values:
            - worker-1

In [ ]:
kubectl apply -f pv-worker1.yaml

create PVC

In [ ]:
vim pvc-worker1.yaml

In [ ]:
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: pvc-master1

spec:
  accessModes:
    - ReadWriteOnce

  resources:
    requests:
      storage: 5Gi

  storageClassName: local-storage

  volumeName: pv-master1

In [ ]:
kubectl apply -f pvc-worker1.yaml

app

In [ ]:
apiVersion: v1
kind: Pod
metadata:
  name: app-worker

spec:
  nodeSelector:
    kubernetes.io/hostname: worker-1

  containers:
  - name: app
    image: nginx

    volumeMounts:
    - mountPath: /data
      name: storage

  volumes:
  - name: storage
    persistentVolumeClaim:
      claimName: pvc-worker1

---

#### master-1 ( nexus )

In [ ]:
sudo mkdir -p /data/nexus
sudo chmod 777 /data/nexus

In [ ]:
vim pv-nexus.yaml

In [ ]:
apiVersion: v1
kind: PersistentVolume
metadata:
  name: pv-nexus-master1

spec:
  capacity:
    storage: 20Gi

  accessModes:
    - ReadWriteOnce

  storageClassName: local-storage

  persistentVolumeReclaimPolicy: Retain

  local:
    path: /data/nexus # <--

  nodeAffinity:
    required:
      nodeSelectorTerms:
      - matchExpressions:
        - key: kubernetes.io/hostname
          operator: In
          values:
            - master-1

In [ ]:
kubectl apply -f pv-nexus.yaml

Verify

In [ ]:
kubectl get pv

PVC

PVC must be in the same name space with app

In [ ]:
 kubectl create namespace nexus

In [ ]:
vim pvc-nexus.yaml

In [ ]:
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: nexus-pvc
  namespace: nexus

spec:
  accessModes:
    - ReadWriteOnce

  resources:
    requests:
      storage: 20Gi

  storageClassName: local-storage

  volumeName: pv-nexus-master1

In [ ]:
kubectl apply -f pvc-nexus.yaml

Add Helm Repo

In [ ]:
helm repo add sonatype https://stevehipwell.github.io/helm-charts/
helm repo update

In [ ]:
vim nexus-values.yaml

In [ ]:
persistence:
  enabled: true
  existingClaim: nexus-pvc

tolerations:
- key: "node-role.kubernetes.io/control-plane"
  operator: "Exists"
  effect: "NoSchedule"

nodeSelector:
  kubernetes.io/hostname: master-1

ingress:
  enabled: true
  ingressClassName: nginx
  hostPath: /
  hostRepo: nexus.voip.local
  annotations:
    nginx.ingress.kubernetes.io/proxy-body-size: "0"
    nginx.ingress.kubernetes.io/proxy-read-timeout: "600"
    nginx.ingress.kubernetes.io/proxy-send-timeout: "600"

nexus:
  nexusPort: 8081   # ← make sure ingress targets this port

readinessProbe:
  initialDelaySeconds: 120    # give Nexus 2 min before first check
  periodSeconds: 15
  failureThreshold: 6
  httpGet:
    path: /service/rest/v1/status
    port: 8081

livenessProbe:
  initialDelaySeconds: 120
  periodSeconds: 15
  failureThreshold: 6
  httpGet:
    path: /service/rest/v1/status
    port: 8081
env:
  - name: INSTALL4J_ADD_VM_PARAMS
    value: "-Xms2703M -Xmx2703M -XX:MaxDirectMemorySize=2703M -XX:+UnlockExperimentalVMOptions -XX:+UseCGroupMemoryLimitForHeap -Djava.util.prefs.userRoot=/nexus-data/javaprefs"

In [ ]:
helm install nexus sonatype/nexus-repository-manager \
  -n nexus \
  -f nexus-values.yaml

In [ ]:
helm upgrade nexus sonatype/nexus-repository-manager \
  -n nexus \
  -f nexus-values.yaml

In [ ]:
kubectl logs -n nexus nexus-nexus-repository-manager-846f46776c-7ljs7